In [1]:
import pandas as pd
import numpy as np
import pickle

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv(
    "data/final_indian_movies_dataset.csv"
)

In [3]:
movies.head()

,title,year,genres,cast,director,overview,rating,poster_url
0,RRR,2022,"Action, Historical, Drama, Adventure","N. T. Rama Rao Jr., Ram Charan, Alia Bhatt, Aj...",S. S. Rajamouli,"Set during British colonial rule in India, the...",7.8,https://image.tmdb.org/t/p/w500/l2ezkzxdlbK1tf...
1,Pushpa: The Rise,2021,"Action, Crime, Drama, Thriller","Allu Arjun, Rashmika Mandanna, Fahadh Faasil",Sukumar,"Pushpa Raj, a daily wage laborer from a poor b...",7.6,https://image.tmdb.org/t/p/w500/uDzje7YJ2Jwbhk...
2,Kantara,2022,"Action, Thriller, Folklore, Drama","Rishab Shetty, Sapthami Gowda, Kishore",Rishab Shetty,In a coastal village deeply connected to divin...,8.2,https://image.tmdb.org/t/p/w500/r7XifzvtezNt31...
3,KGF: Chapter 2,2022,"Action, Crime, Drama","Yash, Sanjay Dutt, Raveena Tandon, Srinidhi Sh...",Prashanth Neel,"After taking control of the Kolar Gold Fields,...",8.3,https://image.tmdb.org/t/p/w500/khNVygolU0TxLI...
4,Jawan,2023,"Action, Thriller, Drama","Shah Rukh Khan, Nayanthara, Vijay Sethupathi, ...",Atlee,A mysterious vigilante hijacks public systems ...,7.0,https://image.tmdb.org/t/p/w500/jFt1gS4BGHlK8x...


In [4]:
movies.shape

(210, 8)

In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   title       210 non-null    object 
 1   year        210 non-null    int64  
 2   genres      210 non-null    object 
 3   cast        210 non-null    object 
 4   director    210 non-null    object 
 5   overview    210 non-null    object 
 6   rating      210 non-null    float64
 7   poster_url  210 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 13.3+ KB


In [6]:
movies.isnull().sum()

title         0
year          0
genres        0
cast          0
director      0
overview      0
rating        0
poster_url    0
dtype: int64

In [7]:
movies['movie_title'] = (

    movies['title']
    + ' ('
    + movies['year'].astype(str)
    + ')'

)

In [8]:
movies['tags'] = (

    movies['overview'] + ' ' +
    movies['overview'] + ' ' +

    movies['genres'] + ' ' +
    movies['genres'] + ' ' +

    movies['cast'] + ' ' +

    movies['director']

)

In [9]:
movies[['movie_title', 'tags']].head(3)

,movie_title,tags
0,RRR (2022),"Set during British colonial rule in India, the..."
1,Pushpa: The Rise (2021),"Pushpa Raj, a daily wage laborer from a poor b..."
2,Kantara (2022),In a coastal village deeply connected to divin...


In [10]:
movies['tags'] = (

    movies['tags']

    .str.replace(
        r'\s+',
        ' ',
        regex=True
    )

    .str.strip()

)

In [11]:
movies[['movie_title', 'tags']].head(2)

,movie_title,tags
0,RRR (2022),"Set during British colonial rule in India, the..."
1,Pushpa: The Rise (2021),"Pushpa Raj, a daily wage laborer from a poor b..."


In [12]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
embeddings = model.encode(

    movies['tags'].tolist(),

    show_progress_bar=True

)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [14]:
pickle.dump(

    embeddings,

    open(
        "embeddings.pkl",
        "wb"
    )

)

In [15]:
indices = pd.Series(

    movies.index,

    index=movies['movie_title']

).drop_duplicates()

In [16]:
def recommend(movie_title, n=10):

    # movie existence check
    if movie_title not in indices:
        return "Movie not found"

    # movie index
    idx = indices[movie_title]

    # target embedding
    movie_embedding = embeddings[idx]

    # cosine similarities
    similarity_scores = cosine_similarity(

        [movie_embedding],
        embeddings

    )[0]

    # sort scores
    similar_movies = sorted(

        list(enumerate(similarity_scores)),

        key=lambda x: x[1],

        reverse=True

    )

    recommendations = []

    for movie in similar_movies:

        movie_index = movie[0]

        # skip same movie
        if movie_index == idx:
            continue

        recommendations.append({

            'movie_title':

                movies.iloc[movie_index][
                    'movie_title'
                ],

            'genres':

                movies.iloc[movie_index][
                    'genres'
                ],

            'similarity_score':

                round(movie[1] * 100, 2)

        })

        if len(recommendations) >= n:
            break

    return recommendations

In [17]:
recommend("Kantara (2022)")

[{'movie_title': 'Kantara: Chapter 1 (2025)',
  'genres': 'Folklore, Action, Fantasy',
  'similarity_score': 73.14},
 {'movie_title': 'Munjya (2024)',
  'genres': 'Horror, Comedy, Folklore',
  'similarity_score': 65.99},
 {'movie_title': 'Virupaksha (2023)',
  'genres': 'Mystery, Horror, Thriller',
  'similarity_score': 65.19},
 {'movie_title': 'RangiTaranga (2015)',
  'genres': 'Mystery, Thriller, Folklore',
  'similarity_score': 64.89},
 {'movie_title': 'Tumbbad (2018)',
  'genres': 'Folklore, Horror, Fantasy',
  'similarity_score': 63.44},
 {'movie_title': 'Hanuman (2024)',
  'genres': 'Superhero, Fantasy, Action',
  'similarity_score': 62.66},
 {'movie_title': 'Chatrapathi (2005)',
  'genres': 'Action, Drama, Emotional',
  'similarity_score': 60.49},
 {'movie_title': 'Awe! (2018)',
  'genres': 'Psychological, Fantasy, Thriller',
  'similarity_score': 60.17},
 {'movie_title': 'Kadaisi Vivasayi (2021)',
  'genres': 'Rural, Drama, Slice of Life',
  'similarity_score': 59.52},
 {'movie

In [18]:
recommend("RRR (2022)")

[{'movie_title': 'Captain Miller (2024)',
  'genres': 'Action, Historical, Drama',
  'similarity_score': 72.34},
 {'movie_title': 'Rang De Basanti (2006)',
  'genres': 'Patriotic, Drama, Political',
  'similarity_score': 68.73},
 {'movie_title': 'Yuva (2004)',
  'genres': 'Political, Drama, Thriller',
  'similarity_score': 65.22},
 {'movie_title': 'Viduthalai Part 1 (2023)',
  'genres': 'Crime, Drama, Political Thriller',
  'similarity_score': 64.86},
 {'movie_title': 'Chatrapathi (2005)',
  'genres': 'Action, Drama, Emotional',
  'similarity_score': 63.37},
 {'movie_title': 'Dasara (2023)',
  'genres': 'Action, Drama, Revenge',
  'similarity_score': 62.64},
 {'movie_title': 'Kaala (2018)',
  'genres': 'Political, Gangster, Drama',
  'similarity_score': 61.03},
 {'movie_title': 'Ayan (2009)',
  'genres': 'Action, Smuggling, Thriller',
  'similarity_score': 60.66},
 {'movie_title': 'Rangasthalam (2018)',
  'genres': 'Action, Drama, Village',
  'similarity_score': 60.21},
 {'movie_title'

In [19]:
recommend("Baahubali: The Beginning (2015)")

'Movie not found'

In [20]:
recommend("Baahubali: The Beginning (2015)")

'Movie not found'

In [21]:
movies[
    movies['movie_title']
    .str.contains(
        'Bah',
        case=False
    )
]['movie_title']

Series([], Name: movie_title, dtype: object)

In [22]:
movies[
    movies['movie_title']
    .str.contains(
        'Baah',
        case=False
    )
]['movie_title']

Series([], Name: movie_title, dtype: object)

In [23]:
recommend("Salaar (2023)")

'Movie not found'

In [24]:
recommend("Pushpa: The Rise (2021)")

[{'movie_title': 'Pudhupettai (2006)',
  'genres': 'Gangster, Crime, Drama',
  'similarity_score': 67.72},
 {'movie_title': 'Businessman (2012)',
  'genres': 'Action, Crime, Drama',
  'similarity_score': 67.42},
 {'movie_title': 'Pokiri (2006)',
  'genres': 'Action, Crime, Thriller',
  'similarity_score': 66.52},
 {'movie_title': 'Kaala (2018)',
  'genres': 'Political, Gangster, Drama',
  'similarity_score': 66.26},
 {'movie_title': 'Ayan (2009)',
  'genres': 'Action, Smuggling, Thriller',
  'similarity_score': 66.1},
 {'movie_title': 'Animal (2023)',
  'genres': 'Action, Crime, Drama, Thriller',
  'similarity_score': 65.64},
 {'movie_title': 'Vedha (2022)',
  'genres': 'Action, Crime, Drama',
  'similarity_score': 64.61},
 {'movie_title': 'Mahaan (2022)',
  'genres': 'Gangster, Drama, Action',
  'similarity_score': 60.56},
 {'movie_title': 'Vada Chennai (2018)',
  'genres': 'Gangster, Crime, Drama',
  'similarity_score': 60.26},
 {'movie_title': 'Jigarthanda (2014)',
  'genres': 'Crim

In [25]:
recommend("KGF Chapter 1 (2018)")

'Movie not found'

In [26]:
recommend("HanuMan (2024)")

'Movie not found'

In [27]:
recommend("Jersey (2019)")

[{'movie_title': 'Dear Comrade (2019)',
  'genres': 'Romance, Drama, Sports',
  'similarity_score': 75.47},
 {'movie_title': 'Majili (2019)',
  'genres': 'Romance, Drama',
  'similarity_score': 75.16},
 {'movie_title': 'Maidaan (2024)',
  'genres': 'Sports, Biography, Drama',
  'similarity_score': 71.68},
 {'movie_title': 'Chak De! India (2007)',
  'genres': 'Sports, Drama, Patriotic',
  'similarity_score': 70.19},
 {'movie_title': 'Sarpatta Parambarai (2021)',
  'genres': 'Sports, Period, Drama',
  'similarity_score': 68.27},
 {'movie_title': 'Bhaag Milkha Bhaag (2013)',
  'genres': 'Biography, Sports, Drama',
  'similarity_score': 67.51},
 {'movie_title': '12th Fail (2023)',
  'genres': 'Drama, Biography',
  'similarity_score': 65.95},
 {'movie_title': 'Chhichhore (2019)',
  'genres': 'Comedy, Drama',
  'similarity_score': 65.56},
 {'movie_title': 'Mahanati (2018)',
  'genres': 'Biography, Drama',
  'similarity_score': 61.92},
 {'movie_title': 'Ala Modalaindi (2011)',
  'genres': 'Ro

In [28]:
recommend("Drishyam (2013)")

'Movie not found'

In [29]:
recommend("Sita Ramam (2022)")

[{'movie_title': 'Dia (2020)',
  'genres': 'Romance, Emotional, Drama',
  'similarity_score': 67.21},
 {'movie_title': 'Hi Nanna (2023)',
  'genres': 'Drama, Romance, Family',
  'similarity_score': 65.6},
 {'movie_title': 'Chatrapathi (2005)',
  'genres': 'Action, Drama, Emotional',
  'similarity_score': 62.36},
 {'movie_title': 'Amaran (2024)',
  'genres': 'War, Biography, Drama',
  'similarity_score': 61.65},
 {'movie_title': 'Athidhi (2007)',
  'genres': 'Action, Thriller, Emotional',
  'similarity_score': 61.45},
 {'movie_title': 'Ugly (2013)',
  'genres': 'Psychological, Crime, Drama',
  'similarity_score': 61.36},
 {'movie_title': 'Ala Modalaindi (2011)',
  'genres': 'Romance, Comedy, Drama',
  'similarity_score': 61.34},
 {'movie_title': 'The Kashmir Files (2022)',
  'genres': 'Drama, Historical',
  'similarity_score': 61.21},
 {'movie_title': 'October (2018)',
  'genres': 'Drama, Emotional, Slice of Life',
  'similarity_score': 61.16},
 {'movie_title': 'Thiruchitrambalam (2022)

In [30]:
movies['movie_title'].tolist()

['RRR (2022)',
 'Pushpa: The Rise (2021)',
 'Kantara (2022)',
 'KGF: Chapter 2 (2022)',
 'Jawan (2023)',
 'Pathaan (2023)',
 'Drishyam 2 (2021)',
 'Minnal Murali (2021)',
 'Sita Ramam (2022)',
 'Hi Nanna (2023)',
 'Master (2021)',
 'Kaithi (2019)',
 'Vikram (2022)',
 'Leo (2023)',
 'Jersey (2019)',
 'Maharaja (2024)',
 'Aavesham (2024)',
 'Manjummel Boys (2024)',
 '777 Charlie (2022)',
 'Kumbalangi Nights (2019)',
 'Soorarai Pottru (2020)',
 'Tumbbad (2018)',
 'Andhadhun (2018)',
 '3 Idiots (2009)',
 'ZNMD (2011)',
 'Ustad Hotel (2012)',
 'Pizza (2012)',
 'Delhi Belly (2011)',
 'Ugly (2013)',
 'October (2018)',
 'Kahaani (2012)',
 'Rock On!! (2008)',
 'Lucifer (2019)',
 'Bhaag Milkha Bhaag (2013)',
 'Queen (2014)',
 '96 (2018)',
 'Vikramarkudu (2006)',
 'Anbe Sivam (2003)',
 'Ayan (2009)',
 'Aadukalam (2011)',
 'Soodhu Kavvum (2013)',
 'Madras (2014)',
 'Kaaka Muttai (2015)',
 'Maan Karate (2014)',
 'Kirik Party (2016)',
 'Dia (2020)',
 'Kavaludaari (2019)',
 'Maya (2015)',
 'Kirik Hit

In [31]:
movies[
    movies['movie_title']
    .str.contains(
        'Push',
        case=False,
        na=False
    )
]['movie_title']

1    Pushpa: The Rise (2021)
Name: movie_title, dtype: object

In [32]:
movies[
    movies['movie_title']
    .str.contains(
        'KGF',
        case=False,
        na=False
    )
]['movie_title']

3    KGF: Chapter 2 (2022)
Name: movie_title, dtype: object

In [33]:
movies[
    movies['movie_title']
    .str.contains(
        'Baah',
        case=False,
        na=False
    )
]['movie_title']

Series([], Name: movie_title, dtype: object)

In [34]:
movies[
    movies['movie_title']
    .str.contains(
        'Bah',
        case=False,
        na=False
    )
]['movie_title']

Series([], Name: movie_title, dtype: object)

In [35]:
len(movies)

210

In [36]:
recommend("Leo (2023)")

[{'movie_title': 'Jailer (2023)',
  'genres': 'Action, Crime, Thriller',
  'similarity_score': 65.41},
 {'movie_title': 'Maharaja (2024)',
  'genres': 'Crime, Thriller, Drama',
  'similarity_score': 62.59},
 {'movie_title': 'Pudhupettai (2006)',
  'genres': 'Gangster, Crime, Drama',
  'similarity_score': 61.39},
 {'movie_title': 'Mangalavaaram (2023)',
  'genres': 'Psychological, Horror, Thriller',
  'similarity_score': 59.97},
 {'movie_title': '96 (2018)',
  'genres': 'Romance, Emotional, Drama',
  'similarity_score': 58.48},
 {'movie_title': 'Vedha (2022)',
  'genres': 'Action, Crime, Drama',
  'similarity_score': 58.38},
 {'movie_title': 'Anand (2004)',
  'genres': 'Romance, Drama, Slice of Life',
  'similarity_score': 58.2},
 {'movie_title': 'Dasara (2023)',
  'genres': 'Action, Drama, Revenge',
  'similarity_score': 57.86},
 {'movie_title': 'Good Night (2023)',
  'genres': 'Comedy, Romance, Slice of Life',
  'similarity_score': 57.7},
 {'movie_title': '7G Rainbow Colony (2004)',
 

In [38]:
recommend("24 (2016)")

[{'movie_title': 'Kalki 2898 AD (2024)',
  'genres': 'Sci-Fi, Action, Mythology',
  'similarity_score': 56.54},
 {'movie_title': 'RangiTaranga (2015)',
  'genres': 'Mystery, Thriller, Folklore',
  'similarity_score': 56.26},
 {'movie_title': 'Swades (2004)',
  'genres': 'Drama, Social',
  'similarity_score': 55.9},
 {'movie_title': 'Rocketry: The Nambi Effect (2022)',
  'genres': 'Biography, Drama, Sci-Fi',
  'similarity_score': 55.39},
 {'movie_title': 'Ratsasan (2018)',
  'genres': 'Psychological, Crime, Thriller',
  'similarity_score': 55.32},
 {'movie_title': 'Manam (2014)',
  'genres': 'Fantasy, Family, Drama',
  'similarity_score': 54.27},
 {'movie_title': 'Super Deluxe (2019)',
  'genres': 'Drama, Dark Comedy, Anthology',
  'similarity_score': 53.82},
 {'movie_title': 'Karthikeya (2014)',
  'genres': 'Mystery, Thriller, Adventure',
  'similarity_score': 52.69},
 {'movie_title': 'Ayan (2009)',
  'genres': 'Action, Smuggling, Thriller',
  'similarity_score': 51.9},
 {'movie_title'